In [1]:
import math
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from based_cnn_model import DiffusionPolicy


/home/kdy1210/ai_architecture_study/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@dataclass
class TrainConfig:
    seed: int = 42
    data_path: str = "./datas/reach_bc_imitation_h15.npz"
    horizon: int = 15
    diffusion_steps: int = 50
    obs_latent: int = 128
    pos_latent: int = 64
    batch_size: int = 64
    num_epochs: int = 20
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    beta_start: float = 1e-4
    beta_end: float = 2e-2
    log_every: int = 200

config = TrainConfig()
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

device

device(type='cuda', index=1)

In [3]:
data = np.load(config.data_path)
observations = data["observations"].astype(np.float32)
action_chunks = data["action_chunks"].astype(np.float32)
action_chunks_t = data["action_chunks_t"].astype(np.float32)

config.horizon = int(data["horizon"])
obs_dim = int(data["obs_dim"])
action_dim = int(data["action_dim"])

print(f"dataset={config.data_path}")
print(f"observations={observations.shape}")
print(f"action_chunks={action_chunks.shape}")
print(f"obs_dim={obs_dim}, action_dim={action_dim}, horizon={config.horizon}")


dataset=./datas/reach_bc_imitation_h15.npz
observations=(1000000, 32)
action_chunks=(1000000, 15, 7)
obs_dim=32, action_dim=7, horizon=15


In [4]:
class ReachDiffusionDataset(Dataset):
    def __init__(self, observations: np.ndarray, action_chunks_t: np.ndarray):
        self.observations = torch.from_numpy(observations)
        self.action_chunks_t = torch.from_numpy(action_chunks_t)

    def __len__(self):
        return len(self.observations)

    def __getitem__(self, idx):
        return {
            "obs": self.observations[idx],
            "action_chunk": self.action_chunks_t[idx],
        }


In [5]:
dataset = ReachDiffusionDataset(observations, action_chunks_t)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)

batch = next(iter(dataloader))
print(batch["obs"].shape, batch["action_chunk"].shape)


torch.Size([64, 32]) torch.Size([64, 7, 15])


In [6]:
print(f"dataset size={len(dataset):,}")
print(f"steps per epoch={len(dataloader):,}")


dataset size=1,000,000
steps per epoch=15,625


In [7]:
model = DiffusionPolicy(
    action_dim=action_dim,
    obs_dim=obs_dim,
    obs_latent=config.obs_latent,
    pos_latent=config.pos_latent,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

betas, alphas, alpha_bars = model.get_b_a_ab(
    start=config.beta_start,
    end=config.beta_end,
    timestep=config.diffusion_steps,
)
alpha_bars = alpha_bars.to(device)

sum(p.numel() for p in model.parameters())


207367

In [8]:
def q_sample_actions(x0: torch.Tensor, t_index: torch.Tensor, alpha_bars: torch.Tensor):
    noise = torch.randn_like(x0)
    alpha_bar_t = alpha_bars[t_index].view(-1, 1, 1)
    xt = torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1.0 - alpha_bar_t) * noise
    return xt, noise

def train_one_epoch(model, dataloader, optimizer, alpha_bars, diffusion_steps, device, log_every, epoch):
    model.train()
    running_loss = 0.0
    progress = tqdm(dataloader, desc=f"epoch {epoch:03d}", leave=False)

    for step, batch in enumerate(progress, start=1):
        obs = batch["obs"].to(device)
        action_chunk = batch["action_chunk"].to(device)

        t_index = torch.randint(0, diffusion_steps, (obs.shape[0],), device=device)
        noisy_action, noise = q_sample_actions(action_chunk, t_index, alpha_bars)

        pred_noise = model(noisy_action, t_index.float(), obs)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        if step % log_every == 0 or step == 1:
            progress.set_postfix(loss=f"{running_loss / step:.6f}")

    return running_loss / max(len(dataloader), 1)

In [9]:
history = []

for epoch in tqdm(range(1, config.num_epochs + 1), desc="training epochs"):
    epoch_loss = train_one_epoch(
        model=model,
        dataloader=dataloader,
        optimizer=optimizer,
        alpha_bars=alpha_bars,
        diffusion_steps=config.diffusion_steps,
        device=device,
        log_every=config.log_every,
        epoch=epoch,
    )
    history.append(epoch_loss)
    print(f"epoch={epoch:03d} mean_loss={epoch_loss:.6f}")


training epochs:   5%|▌         | 1/20 [01:16<24:06, 76.12s/it]

epoch=001 mean_loss=1.302935


training epochs:  10%|█         | 2/20 [02:24<21:26, 71.46s/it]

epoch=002 mean_loss=0.999806


training epochs:  15%|█▌        | 3/20 [03:42<21:04, 74.40s/it]

...

epoch=018 mean_loss=1.000016


training epochs:  95%|█████████▌| 19/20 [22:32<01:11, 71.13s/it]

epoch=019 mean_loss=0.999961


training epochs: 100%|██████████| 20/20 [23:42<00:00, 71.13s/it]

epoch=020 mean_loss=0.999855


In [10]:
@torch.no_grad()
def sample_action_chunk(model, obs: torch.Tensor, horizon: int, action_dim: int, alpha_bars: torch.Tensor, beta_start: float, beta_end: float, diffusion_steps: int):
    model.eval()
    x = torch.randn(1, action_dim, horizon, device=obs.device)

    betas, alphas, _ = model.get_b_a_ab(beta_start, beta_end, diffusion_steps)
    betas = betas.to(obs.device)
    alphas = alphas.to(obs.device)

    for step in reversed(range(diffusion_steps)):
        t_batch = torch.full((1,), float(step), device=obs.device)
        pred_noise = model(x, t_batch, obs)

        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]
        coef = (1.0 - alpha_t) / torch.sqrt(1.0 - alpha_bar_t)
        x = (x - coef * pred_noise) / torch.sqrt(alpha_t)

        if step > 0:
            x = x + torch.sqrt(betas[step]) * torch.randn_like(x)

    return x.squeeze(0).transpose(0, 1)


In [11]:
test_obs = torch.from_numpy(observations[0]).unsqueeze(0).to(device)
gt_chunk = torch.from_numpy(action_chunks[0])
print(test_obs.shape, gt_chunk.shape)


torch.Size([1, 32]) torch.Size([15, 7])


In [12]:
sampled_chunk = sample_action_chunk(
    model=model,
    obs=test_obs,
    horizon=config.horizon,
    action_dim=action_dim,
    alpha_bars=alpha_bars,
    beta_start=config.beta_start,
    beta_end=config.beta_end,
    diffusion_steps=config.diffusion_steps,
)
sampled_chunk.shape


torch.Size([15, 7])

In [13]:
print("ground truth chunk[0]")
print(gt_chunk)


ground truth chunk[0]
tensor([[  17.6552,   12.1108,   28.4821,   -9.6592,    0.3618,   24.7958,
           -3.3527],
        [  37.1561,   33.6210,   52.4586,  -22.2621,    6.3087,   47.0246,
          -12.6862],
        [  54.6105,   46.3145,   73.1615,  -36.7046,    8.3309,   65.2239,
          -22.6299],
        [  63.4201,   50.3302,   82.7867,  -48.4775,    5.0422,   80.4293,
          -33.8365],
        [  65.1505,   52.5929,   88.9083,  -56.5057,    1.6086,   92.4904,
          -40.7903],
        [  63.1479,   53.4615,   93.5099,  -63.7312,   -1.2676,  101.8695,
          -44.4581],
        [  59.6954,   51.6785,   97.7796,  -71.8862,   -3.9469,  109.6363,
          -47.3215],
        [  55.6391,   49.2588,  102.1410,  -80.9340,   -6.3348,  116.1670,
          -49.5922],
        [  51.6252,   46.8609,  106.6469,  -90.3805,   -8.4612,  121.8657,
          -51.6328],
        [  48.3217,   45.1590,  110.5054, -100.1055,  -10.3015,  127.6983,
          -53.9696],
        [  44.4640

In [14]:
print("sampled chunk[0]")
print(sampled_chunk.detach().cpu())


sampled chunk[0]
tensor([[ 1.0897e-01,  1.4660e+00, -1.4574e+00, -1.6773e+00, -4.2412e-01,
          8.2359e-01,  2.4946e+00],
        [ 5.4914e-01,  1.0111e-01,  2.7259e+00, -2.2344e+00,  1.5601e-01,
         -1.7849e-01, -3.5911e-01],
        [-1.1728e+00,  4.3624e-01, -5.3688e-01, -1.6911e+00, -1.5965e-01,
         -1.1340e+00,  8.2595e-01],
        [ 1.0911e+00, -4.3777e-01,  3.2605e+00,  2.3977e-01, -2.8638e+00,
          7.0757e-01,  6.5930e-01],
        [-1.5902e+00,  2.9539e+00,  6.2552e-01,  1.7295e+00, -8.6609e-01,
         -1.9934e-03, -1.2209e+00],
        [ 1.4451e+00, -1.2216e+00,  1.4657e+00, -7.8246e-01,  1.5786e+00,
          8.5463e-01, -1.1262e+00],
        [-2.6791e+00, -2.4801e+00,  1.4484e+00,  6.4274e-01,  1.5328e+00,
         -3.0470e+00,  1.7734e+00],
        [-8.4313e-01,  2.8551e-01,  5.4636e-01,  8.3060e-01, -1.3683e+00,
          8.0188e-01, -1.6077e+00],
        [-1.3530e+00, -3.8232e-01,  2.0500e+00, -4.1486e+00, -2.9502e-01,
         -2.5140e-01,  2.1229